# 14: Word2Vec Intuition - Learning Word Meanings

## How Do Machines Learn That "King - Man + Woman = Queen"?

Word2Vec (2013) was a breakthrough: it showed that **word meanings can be captured by prediction tasks**.

The core insight:
> "You shall know a word by the company it keeps" - J.R. Firth, 1957

Words that appear in similar contexts have similar meanings.

### The Web Dev Analogy

Think of Word2Vec like learning user preferences from behavior:
- You don't ask users what they like
- You observe: "Users who clicked X also clicked Y"
- Similar users cluster together in "preference space"

Word2Vec does the same: it observes word co-occurrences and learns that similar words appear in similar contexts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
torch.manual_seed(42)

print("Ready to understand Word2Vec!")

## 1. The Distributional Hypothesis

Words with similar meanings appear in similar contexts. Let's see this in action.

In [ ]:
# Sample sentences - notice how similar words have similar contexts
sentences = [
    "the king sits on the throne",
    "the queen sits on the throne",
    "the king wears a crown",
    "the queen wears a crown",
    "the man works in the field",
    "the woman works in the field",
    "the boy plays in the garden",
    "the girl plays in the garden",
    "the prince is young royalty",
    "the princess is young royalty",
    "the king rules the kingdom",
    "the queen rules the kingdom",
]

# Look at contexts for different words
def get_contexts(sentences, target_word, window=2):
    """Find words that appear near the target word."""
    contexts = []
    for sentence in sentences:
        words = sentence.lower().split()
        for i, word in enumerate(words):
            if word == target_word:
                start = max(0, i - window)
                end = min(len(words), i + window + 1)
                context = [words[j] for j in range(start, end) if j != i]
                contexts.extend(context)
    return contexts

print("Contexts for 'king':")
print(get_contexts(sentences, 'king'))
print("\nContexts for 'queen':")
print(get_contexts(sentences, 'queen'))
print("\nNotice how similar they are!")

## 2. Two Word2Vec Architectures

Word2Vec comes in two flavors:

### Skip-gram: Predict context from word
- Given: "king"
- Predict: "the", "sits", "on", "throne"

### CBOW (Continuous Bag of Words): Predict word from context
- Given: "the", "sits", "on", "throne"
- Predict: "king"

Let's visualize these!

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Skip-gram visualization
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)

# Center word
ax1.add_patch(plt.Rectangle((4, 4), 2, 2, fill=True, color='royalblue', alpha=0.8))
ax1.text(5, 5, 'king', ha='center', va='center', fontsize=14, fontweight='bold', color='white')

# Context words
context_positions = [(1, 7), (7, 7), (1, 2), (7, 2)]
context_words = ['the', 'sits', 'on', 'throne']
for (x, y), word in zip(context_positions, context_words):
    ax1.add_patch(plt.Rectangle((x, y), 2, 1.5, fill=True, color='coral', alpha=0.7))
    ax1.text(x+1, y+0.75, word, ha='center', va='center', fontsize=12)
    ax1.annotate('', xy=(x+1, y+0.75), xytext=(5, 5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax1.set_title('Skip-gram: Predict Context from Word', fontsize=14, fontweight='bold')
ax1.text(5, 9, 'Input: center word', ha='center', fontsize=11)
ax1.text(5, 0.5, 'Output: surrounding words', ha='center', fontsize=11)
ax1.axis('off')

# CBOW visualization
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)

# Center word (output)
ax2.add_patch(plt.Rectangle((4, 4), 2, 2, fill=True, color='coral', alpha=0.8))
ax2.text(5, 5, 'king', ha='center', va='center', fontsize=14, fontweight='bold', color='white')

# Context words (input)
for (x, y), word in zip(context_positions, context_words):
    ax2.add_patch(plt.Rectangle((x, y), 2, 1.5, fill=True, color='royalblue', alpha=0.7))
    ax2.text(x+1, y+0.75, word, ha='center', va='center', fontsize=12, color='white')
    ax2.annotate('', xy=(5, 5), xytext=(x+1, y+0.75),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax2.set_title('CBOW: Predict Word from Context', fontsize=14, fontweight='bold')
ax2.text(5, 9, 'Input: surrounding words', ha='center', fontsize=11)
ax2.text(5, 0.5, 'Output: center word', ha='center', fontsize=11)
ax2.axis('off')

plt.tight_layout()
plt.show()

print("Skip-gram is better for rare words, CBOW is faster for frequent words.")

## 3. The Skip-gram Architecture

Let's implement a simple Skip-gram model to see how it works.

The architecture is surprisingly simple:
1. **Input**: One-hot encoded word
2. **Hidden layer**: The embedding (what we want!)
3. **Output**: Probability distribution over all words

In [ ]:
# Build vocabulary
all_words = []
for sentence in sentences:
    all_words.extend(sentence.lower().split())
vocab = list(set(all_words))
vocab_size = len(vocab)

word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

print(f"Vocabulary size: {vocab_size}")
print(f"Words: {vocab}")

In [ ]:
# Create training data for Skip-gram
def create_skipgram_data(sentences, window_size=2):
    """Create (center_word, context_word) pairs."""
    data = []
    for sentence in sentences:
        words = sentence.lower().split()
        for i, center_word in enumerate(words):
            # Look at words within the window
            for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
                if i != j:  # Don't predict the word itself
                    context_word = words[j]
                    data.append((word_to_idx[center_word], word_to_idx[context_word]))
    return data

training_data = create_skipgram_data(sentences, window_size=2)

print(f"Training pairs: {len(training_data)}")
print("\nFirst 10 pairs:")
for center_idx, context_idx in training_data[:10]:
    print(f"  {idx_to_word[center_idx]:10} -> {idx_to_word[context_idx]}")

In [ ]:
# Simple Skip-gram model
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # Two embedding matrices:
        # - embeddings: for center words (this is what we keep!)
        # - context_embeddings: for context words
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
    def forward(self, center_word, context_word):
        # Get embeddings
        center_embed = self.embeddings(center_word)      # (batch, embed_dim)
        context_embed = self.context_embeddings(context_word)  # (batch, embed_dim)
        
        # Dot product gives similarity score
        score = torch.sum(center_embed * context_embed, dim=1)
        return score
    
    def get_embedding(self, word_idx):
        """Get the learned embedding for a word."""
        return self.embeddings(torch.tensor([word_idx])).detach().numpy()[0]

# Create model
embedding_dim = 10
model = SkipGram(vocab_size, embedding_dim)

print(f"Model created with {embedding_dim}-dimensional embeddings")
print(f"Embedding matrix shape: {model.embeddings.weight.shape}")

In [ ]:
# Training with negative sampling (simplified)
# Instead of softmax over all words, we sample negative examples

def train_skipgram(model, training_data, epochs=100, lr=0.01, num_negative=5):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []
    
    for epoch in range(epochs):
        total_loss = 0
        np.random.shuffle(training_data)
        
        for center_idx, context_idx in training_data:
            # Positive sample
            center = torch.tensor([center_idx])
            context = torch.tensor([context_idx])
            
            pos_score = model(center, context)
            pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-10)
            
            # Negative samples (random words that shouldn't be in context)
            neg_loss = 0
            for _ in range(num_negative):
                neg_idx = np.random.randint(0, vocab_size)
                while neg_idx == context_idx:
                    neg_idx = np.random.randint(0, vocab_size)
                neg_context = torch.tensor([neg_idx])
                neg_score = model(center, neg_context)
                neg_loss += -torch.log(torch.sigmoid(-neg_score) + 1e-10)
            
            loss = pos_loss + neg_loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        losses.append(total_loss / len(training_data))
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}, Loss: {losses[-1]:.4f}")
    
    return losses

print("Training Skip-gram model...")
losses = train_skipgram(model, training_data, epochs=100)

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Skip-gram Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## 4. The Magic: King - Man + Woman = Queen

This is the most famous result from Word2Vec: **word arithmetic**!

Because embeddings capture meaning geometrically:
- king - man captures "royalty"
- Adding woman gives us "female royalty" = queen

In [ ]:
# Get learned embeddings
def get_word_vector(word):
    return model.get_embedding(word_to_idx[word])

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-10)

def find_most_similar(vector, top_k=5):
    """Find words most similar to a vector."""
    similarities = []
    for word in vocab:
        word_vec = get_word_vector(word)
        sim = cosine_similarity(vector, word_vec)
        similarities.append((word, sim))
    return sorted(similarities, key=lambda x: x[1], reverse=True)[:top_k]

# Word arithmetic: king - man + woman = ?
king = get_word_vector('king')
man = get_word_vector('man')
woman = get_word_vector('woman')

result = king - man + woman

print("king - man + woman = ?")
print("\nMost similar words to result:")
for word, sim in find_most_similar(result):
    print(f"  {word}: {sim:.3f}")

In [ ]:
# Visualize the word analogy
from sklearn.decomposition import PCA

# Get all embeddings
all_embeddings = np.array([get_word_vector(word) for word in vocab])

# Reduce to 2D
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(all_embeddings)

# Create word to 2D mapping
word_to_2d = {word: embeddings_2d[i] for i, word in enumerate(vocab)}

plt.figure(figsize=(12, 10))

# Plot all words
for word in vocab:
    x, y = word_to_2d[word]
    plt.scatter(x, y, s=100, alpha=0.6)
    plt.annotate(word, (x + 0.02, y + 0.02), fontsize=11)

# Highlight the analogy words
analogy_words = ['king', 'queen', 'man', 'woman']
for word in analogy_words:
    if word in word_to_2d:
        x, y = word_to_2d[word]
        plt.scatter(x, y, s=300, c='red', alpha=0.5, edgecolors='black', linewidth=2)

# Draw the analogy vectors
if all(w in word_to_2d for w in analogy_words):
    # king -> queen (royalty direction)
    k = word_to_2d['king']
    q = word_to_2d['queen']
    plt.annotate('', xy=q, xytext=k,
                arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    
    # man -> woman (gender direction)
    m = word_to_2d['man']
    w = word_to_2d['woman']
    plt.annotate('', xy=w, xytext=m,
                arrowprops=dict(arrowstyle='->', color='green', lw=2))

plt.xlabel('PCA Dimension 1')
plt.ylabel('PCA Dimension 2')
plt.title('Word Embeddings: Visualizing Analogies\n(king - man + woman = queen)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

print("Blue arrow: king -> queen (same gender difference as man -> woman in green)")

## 5. Why Does This Work?

Let's build intuition for why word arithmetic works.

In [ ]:
# Visualize the concept with a cleaner example
fig, ax = plt.subplots(figsize=(10, 10))

# Idealized embedding space
# Imagine 2 dimensions: royalty (x) and gender (y)
ideal_positions = {
    'man': (0, 0),
    'woman': (0, 1),
    'king': (1, 0),
    'queen': (1, 1),
    'prince': (0.8, 0.3),
    'princess': (0.8, 0.7),
    'boy': (0.2, 0.2),
    'girl': (0.2, 0.8),
}

# Plot points
for word, (x, y) in ideal_positions.items():
    color = 'blue' if y < 0.5 else 'red'
    ax.scatter(x, y, s=200, c=color, alpha=0.7)
    ax.annotate(word, (x + 0.03, y + 0.03), fontsize=14)

# Draw the parallelogram
ax.plot([0, 0, 1, 1, 0], [0, 1, 1, 0, 0], 'k--', alpha=0.3, lw=2)

# Gender direction arrow
ax.annotate('', xy=(0, 1), xytext=(0, 0),
           arrowprops=dict(arrowstyle='->', color='green', lw=3))
ax.text(-0.15, 0.5, 'Gender\nDirection', ha='center', va='center', fontsize=12, color='green')

# Royalty direction arrow
ax.annotate('', xy=(1, 0), xytext=(0, 0),
           arrowprops=dict(arrowstyle='->', color='purple', lw=3))
ax.text(0.5, -0.1, 'Royalty Direction', ha='center', va='center', fontsize=12, color='purple')

# The arithmetic
ax.annotate('', xy=(1, 1), xytext=(1, 0),
           arrowprops=dict(arrowstyle='->', color='orange', lw=3))

ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('Royalty', fontsize=14)
ax.set_ylabel('Gender', fontsize=14)
ax.set_title('Why Word Arithmetic Works\n\nking - man + woman = king + (woman - man) = king + gender_diff = queen', 
            fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# Legend
ax.scatter([], [], c='blue', label='Male', s=100)
ax.scatter([], [], c='red', label='Female', s=100)
ax.legend(loc='upper left')

plt.show()

print("The math: king + (woman - man) = king + 'female direction' = queen")

In [ ]:
# More analogies (if our tiny corpus captured them)
def analogy(word_a, word_b, word_c):
    """a is to b as c is to ?"""
    vec_a = get_word_vector(word_a)
    vec_b = get_word_vector(word_b)
    vec_c = get_word_vector(word_c)
    
    # a - b + c = ?
    result = vec_a - vec_b + vec_c
    
    print(f"{word_a} - {word_b} + {word_c} = ?")
    print(f"({word_a} is to {word_b} as {word_c} is to ?)")
    print("\nTop matches:")
    for word, sim in find_most_similar(result, top_k=3):
        if word not in [word_a, word_b, word_c]:
            print(f"  {word}: {sim:.3f}")
    print()

# Try some analogies
analogy('king', 'man', 'woman')
analogy('prince', 'boy', 'girl')

## 6. How Word2Vec Actually Learns

Let's trace through what happens during training.

In [ ]:
# Visualize the learning process
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: The task
ax1 = axes[0]
ax1.text(0.5, 0.9, 'The Skip-gram Task', ha='center', fontsize=14, fontweight='bold',
        transform=ax1.transAxes)
ax1.text(0.5, 0.7, 'Input: "king"', ha='center', fontsize=12, transform=ax1.transAxes)
ax1.text(0.5, 0.5, 'Predict nearby words:', ha='center', fontsize=12, transform=ax1.transAxes)
ax1.text(0.5, 0.35, '"the", "sits", "throne", "crown"', ha='center', fontsize=11,
        transform=ax1.transAxes, style='italic')
ax1.text(0.5, 0.15, 'Learn: words appearing together\nshould have similar embeddings',
        ha='center', fontsize=10, transform=ax1.transAxes)
ax1.axis('off')

# Panel 2: The architecture
ax2 = axes[1]
ax2.text(0.5, 0.95, 'The Architecture', ha='center', fontsize=14, fontweight='bold',
        transform=ax2.transAxes)

# Input layer
ax2.add_patch(plt.Rectangle((0.1, 0.65), 0.2, 0.2, fill=True, color='lightblue', alpha=0.8))
ax2.text(0.2, 0.75, 'One-hot\n"king"', ha='center', va='center', fontsize=10)

# Hidden layer (embedding)
ax2.add_patch(plt.Rectangle((0.4, 0.65), 0.2, 0.2, fill=True, color='lightgreen', alpha=0.8))
ax2.text(0.5, 0.75, 'Embedding\n(10 dims)', ha='center', va='center', fontsize=10)

# Output layer
ax2.add_patch(plt.Rectangle((0.7, 0.65), 0.2, 0.2, fill=True, color='lightyellow', alpha=0.8))
ax2.text(0.8, 0.75, 'Softmax\n(vocab)', ha='center', va='center', fontsize=10)

# Arrows
ax2.annotate('', xy=(0.4, 0.75), xytext=(0.3, 0.75),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax2.annotate('', xy=(0.7, 0.75), xytext=(0.6, 0.75),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax2.text(0.5, 0.4, 'The embedding layer IS\nwhat we want to learn!', ha='center', fontsize=11,
        transform=ax2.transAxes, fontweight='bold', color='green')
ax2.text(0.5, 0.2, 'After training, throw away\nthe output layer.', ha='center', fontsize=10,
        transform=ax2.transAxes)
ax2.axis('off')

# Panel 3: What's learned
ax3 = axes[2]
ax3.text(0.5, 0.95, 'What Gets Learned', ha='center', fontsize=14, fontweight='bold',
        transform=ax3.transAxes)
ax3.text(0.5, 0.75, 'Words in similar contexts', ha='center', fontsize=12,
        transform=ax3.transAxes)
ax3.text(0.5, 0.6, 'get similar embeddings', ha='center', fontsize=12,
        transform=ax3.transAxes)
ax3.text(0.5, 0.4, '"king" and "queen" both appear near:', ha='center', fontsize=11,
        transform=ax3.transAxes)
ax3.text(0.5, 0.25, '"throne", "crown", "rules"', ha='center', fontsize=11,
        transform=ax3.transAxes, style='italic')
ax3.text(0.5, 0.1, 'So they end up close in\nembedding space!', ha='center', fontsize=11,
        transform=ax3.transAxes, fontweight='bold', color='blue')
ax3.axis('off')

plt.tight_layout()
plt.show()

## 7. Comparing Skip-gram vs CBOW

In [ ]:
# Create a comparison table
comparison_data = {
    'Aspect': ['Task', 'Input', 'Output', 'Speed', 'Rare Words', 'Best For'],
    'Skip-gram': [
        'Predict context from word',
        'Center word',
        'Context words',
        'Slower (more training pairs)',
        'Better (each word is center)',
        'Small datasets, rare words'
    ],
    'CBOW': [
        'Predict word from context',
        'Context words',
        'Center word',
        'Faster (fewer computations)',
        'Worse (rare words smoothed)',
        'Large datasets, frequent words'
    ]
}

# Display as formatted text
print("Skip-gram vs CBOW Comparison")
print("=" * 70)
for i, aspect in enumerate(comparison_data['Aspect']):
    print(f"\n{aspect}:")
    print(f"  Skip-gram: {comparison_data['Skip-gram'][i]}")
    print(f"  CBOW:      {comparison_data['CBOW'][i]}")

## 8. Key Insights for Intuition

In [ ]:
# Visualize the key insights
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Insight 1: Context defines meaning
ax1 = axes[0, 0]
ax1.text(0.5, 0.9, 'Insight 1: Context Defines Meaning', ha='center', fontsize=13, 
        fontweight='bold', transform=ax1.transAxes)
ax1.text(0.5, 0.65, '"The ___ barked loudly"', ha='center', fontsize=14, 
        transform=ax1.transAxes, style='italic')
ax1.text(0.5, 0.5, 'What fits? dog, puppy, hound...', ha='center', fontsize=12, 
        transform=ax1.transAxes)
ax1.text(0.5, 0.35, 'Words that fit similar blanks', ha='center', fontsize=12, 
        transform=ax1.transAxes)
ax1.text(0.5, 0.2, 'get similar embeddings!', ha='center', fontsize=12, 
        transform=ax1.transAxes, fontweight='bold', color='blue')
ax1.axis('off')

# Insight 2: Dimensions capture features
ax2 = axes[0, 1]
ax2.text(0.5, 0.9, 'Insight 2: Dimensions = Features', ha='center', fontsize=13, 
        fontweight='bold', transform=ax2.transAxes)
ax2.text(0.5, 0.7, 'Each dimension might capture:', ha='center', fontsize=12, 
        transform=ax2.transAxes)
features = ['Gender (male <-> female)', 'Royalty (common <-> royal)', 
           'Age (young <-> old)', 'Size (small <-> big)']
for i, feat in enumerate(features):
    ax2.text(0.5, 0.55 - i*0.12, f'* {feat}', ha='center', fontsize=11, 
            transform=ax2.transAxes)
ax2.text(0.5, 0.1, '(Emergent, not designed!)', ha='center', fontsize=11, 
        transform=ax2.transAxes, style='italic')
ax2.axis('off')

# Insight 3: Relationships are vectors
ax3 = axes[1, 0]
ax3.text(0.5, 0.9, 'Insight 3: Relationships = Vectors', ha='center', fontsize=13, 
        fontweight='bold', transform=ax3.transAxes)
ax3.text(0.5, 0.7, 'The difference vector captures the relationship:', ha='center', fontsize=11, 
        transform=ax3.transAxes)
relationships = [
    'queen - king = female - male',
    'Paris - France = Berlin - Germany',
    'running - run = swimming - swim'
]
for i, rel in enumerate(relationships):
    ax3.text(0.5, 0.5 - i*0.12, rel, ha='center', fontsize=11, 
            transform=ax3.transAxes, family='monospace')
ax3.axis('off')

# Insight 4: Geometric operations have meaning
ax4 = axes[1, 1]
ax4.text(0.5, 0.9, 'Insight 4: Geometry = Semantics', ha='center', fontsize=13, 
        fontweight='bold', transform=ax4.transAxes)
ops = [
    ('Distance', 'Similarity (close = similar)'),
    ('Direction', 'Relationship type'),
    ('Clustering', 'Categories (animals, verbs, etc.)'),
    ('Arithmetic', 'Analogies!')
]
for i, (op, meaning) in enumerate(ops):
    ax4.text(0.3, 0.7 - i*0.15, f'{op}:', ha='right', fontsize=11, 
            transform=ax4.transAxes, fontweight='bold')
    ax4.text(0.35, 0.7 - i*0.15, meaning, ha='left', fontsize=11, 
            transform=ax4.transAxes)
ax4.axis('off')

plt.tight_layout()
plt.show()

## Check Your Understanding

1. What is the distributional hypothesis and how does Word2Vec use it?
2. What's the difference between Skip-gram and CBOW?
3. Why does "king - man + woman = queen" work mathematically?
4. What part of the neural network do we keep after training?
5. Why is negative sampling used instead of full softmax?

## Summary

**Word2Vec** learns word embeddings by predicting context:

**Two Architectures:**
- **Skip-gram**: Predict context words from center word
- **CBOW**: Predict center word from context words

**Key Insights:**
- Words in similar contexts get similar embeddings
- Embedding dimensions capture semantic features
- Word arithmetic works because relationships are consistent vectors

**The Magic Formula:**
```
king - man + woman = queen
```
Because `(woman - man)` captures the "gender direction", and adding it to `king` moves to `queen`!

**Next up**: Visualizing embeddings with t-SNE and PCA!